# Partitioning

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [4]:
spark_path = "E:\\Spark\\spark-3.2.0-bin-hadoop2.7"

In [5]:
import findspark
findspark.init(spark_path)

In [6]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [7]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "10g")
    .master("local[*]")
    .appName("Partitioning")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

E:\Spark\spark-3.2.0-bin-hadoop2.7\python\pyspark\context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


In [8]:
listening_activity_file = "../data/partitioning/raw/Spotify_Listening_Activity.csv"
df_listening_actv = spark.read.csv(listening_activity_file, header=True, inferSchema=True)
df_listening_actv.show(5, False)

+-----------+-------+--------------------------+---------------+
|activity_id|song_id|listen_date               |listen_duration|
+-----------+-------+--------------------------+---------------+
|1          |12     |2023-06-27 10:15:47.008867|69             |
|2          |44     |2023-06-27 10:15:47.008867|300            |
|3          |75     |2023-06-27 10:15:47.008867|73             |
|4          |48     |2023-06-27 10:15:47.008867|105            |
|5          |10     |2023-06-27 10:15:47.008867|229            |
+-----------+-------+--------------------------+---------------+
only showing top 5 rows



In [9]:
df_listening_actv = (
    df_listening_actv
    .withColumnRenamed("listen_date", "listen_time")
    .withColumn("listen_date", F.to_date("listen_time", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    .withColumn("listen_hour", F.hour("listen_time"))
)

df_listening_actv.show(5, False)
df_listening_actv.printSchema()
df_listening_actv.count()

+-----------+-------+--------------------------+---------------+-----------+-----------+
|activity_id|song_id|listen_time               |listen_duration|listen_date|listen_hour|
+-----------+-------+--------------------------+---------------+-----------+-----------+
|1          |12     |2023-06-27 10:15:47.008867|69             |2023-06-27 |10         |
|2          |44     |2023-06-27 10:15:47.008867|300            |2023-06-27 |10         |
|3          |75     |2023-06-27 10:15:47.008867|73             |2023-06-27 |10         |
|4          |48     |2023-06-27 10:15:47.008867|105            |2023-06-27 |10         |
|5          |10     |2023-06-27 10:15:47.008867|229            |2023-06-27 |10         |
+-----------+-------+--------------------------+---------------+-----------+-----------+
only showing top 5 rows

root
 |-- activity_id: integer (nullable = true)
 |-- song_id: integer (nullable = true)
 |-- listen_time: string (nullable = true)
 |-- listen_duration: integer (nullable = 

11779

## Partitioning By `listen_date`

Let's say we want to **analyse the listening behaviours of user over time**. If we're given the complete dataset (with no partitions), Spark would scan the whole dataset for finding a particular date (similar to the bookshelf analogy where you would scan the entire bookself for finding a book if it is not organized). Given that our usecase needs analysis by date, partitioning (creating folders) on date would help Spark pin point to the exact folder. This makes searching very easy and Spark doesn't scan the entire dataset.  

In [10]:
# Partitioning listening activity by the listen date
(
    df_listening_actv
    .write
    .partitionBy("listen_date")
    .mode("overwrite")
    .parquet("./output/listening_activity_pt")
)

In [11]:
# Reading from a partitioned dataset
df_partitioned = spark.read.parquet("./output/listening_activity_pt")
print(f"Number of partitions: {df_partitioned.rdd.getNumPartitions()}")

Number of partitions: 12


In [12]:
# Partitioning listening activity by the listen date in CSV format
# is listen_date part of the actual data file?
(
    df_listening_actv
    .write
    .partitionBy("listen_date")
    .mode("overwrite")
    .csv("./output/listening_activity_pt_csv", header=True)
)

In [13]:
spark.read.csv("./output/listening_activity_pt_csv/listen_date=2023-04-25", header=True).show(5, False)

+-----------+-------+--------------------------+---------------+-----------+
|activity_id|song_id|listen_time               |listen_duration|listen_hour|
+-----------+-------+--------------------------+---------------+-----------+
|3467       |33     |2023-04-25 10:15:47.019288|138            |10         |
|3468       |43     |2023-04-25 10:15:47.019288|250            |10         |
|3469       |66     |2023-04-25 10:15:47.019288|222            |10         |
|3470       |36     |2023-04-25 10:15:47.019288|116            |10         |
|3471       |54     |2023-04-25 10:15:47.019288|192            |10         |
+-----------+-------+--------------------------+---------------+-----------+
only showing top 5 rows



In [14]:
spark.read.csv("./output/listening_activity_pt_csv", header=True).show(5, False)

+-----------+-------+--------------------------+---------------+-----------+-----------+
|activity_id|song_id|listen_time               |listen_duration|listen_hour|listen_date|
+-----------+-------+--------------------------+---------------+-----------+-----------+
|4456       |16     |2023-07-18 10:15:47.023264|151            |10         |2023-07-18 |
|4457       |65     |2023-07-18 10:15:47.023264|181            |10         |2023-07-18 |
|4458       |60     |2023-07-18 10:15:47.023264|280            |10         |2023-07-18 |
|4459       |3      |2023-07-18 10:15:47.023264|249            |10         |2023-07-18 |
|4460       |45     |2023-07-18 10:15:47.023264|130            |10         |2023-07-18 |
+-----------+-------+--------------------------+---------------+-----------+-----------+
only showing top 5 rows



## Partition Pruning

In [15]:
df_listening_actv_pt_pruned = spark.read.parquet("./output/listening_activity_pt")
df_listening_actv_pt_pruned.filter("listen_date = '2023-04-25'").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [activity_id#246,song_id#247,listen_time#248,listen_duration#249,listen_hour#250,listen_date#251] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/E:/PySpark/spark_performance_tuning/output/listening_activity_pt], PartitionFilters: [isnotnull(listen_date#251), (listen_date#251 = 2023-04-25)], PushedFilters: [], ReadSchema: struct<activity_id:int,song_id:int,listen_time:string,listen_duration:int,listen_hour:int>




## What Problems Does Partitioning Solve? 
1. `Fast Search (Query Performance)`: Spark will only process the relevant partition instead of the entire dataset (example above). This greatly reduces I/O and query execution time. 
2. `Parallelism / Resource Utilization`: Each core processes 1 partition; More number of partitions, more is the parallelism; again this does not mean we forcefully increase the number of partitions. Each partition should be `128MB` in size. 


# Partitioning Examples
1. Single/multi level partitioning
2. Using `repartition`/`coalesce` with `partitionBy` (controlling number of files inside each partition): 
    - `parititionBy` affects how data is laid out in the storage and is going to ensure that the output directory is organized into subdirectories based on the `value` given in `partitionBy`.  
    - Number of files in each `value` directory of `partitionBy` depends on the number supplied in the `repartition`/`coalesce`.

#### 1. Single/multi level partitioning

In [16]:
(
    df_listening_actv
    .write
    .mode("overwrite")
    .partitionBy("listen_date", "listen_hour")
    .parquet("./output/listening_activity_pt_2")
)

In [17]:
(
    df_listening_actv
    .write
    .mode("overwrite")
    .partitionBy("listen_hour", "listen_date")
    .parquet("./output/listening_activity_pt_3")
)

#### 2. Using `repartition`/`coalesce` with `partitionBy`

In [18]:
# repartition can be used to create desired number of partition files in each partition folder
(
    df_listening_actv
    .repartition(3)
    .write
    .mode("overwrite")
    .partitionBy("listen_date")
    .parquet("./output/listening_activity_pt_4")
)

In [19]:
# coalesce reduces the number of partitions in a DataFrame. 
# It avoids full shuffle. Instead of creating new partitions, it shuffles the data using default Hash Partitioner, 
# and adjusts into existing partitions, this means it can only decrease the number of partitions.

(
    df_listening_actv
    .coalesce(3)
    .write
    .mode("overwrite")
    .partitionBy("listen_date")
    .parquet("./output/listening_activity_pt_5")
)

## Experimenting With `spark.sql.files.maxPartitionBytes`

In [20]:
spark.stop()
spark = SparkSession.builder.appName("Test spark.sql.files.maxPartitionBytes").getOrCreate()

df_default = spark.read.csv("../data/partitioning/raw/Spotify_Listening_Activity.csv", header=True, inferSchema=True)
default_partitions = df_default.rdd.getNumPartitions()
print(f"Number of partitions with default maxPartitionBytes: {default_partitions}")


Number of partitions with default maxPartitionBytes: 1


E:\Spark\spark-3.2.0-bin-hadoop2.7\python\pyspark\context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


In [21]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "1000")

df_modified = spark.read.csv("../data/partitioning/raw/Spotify_Listening_Activity.csv", header=True, inferSchema=True)
modified_partitions = df_modified.rdd.getNumPartitions()
print(f"Number of partitions with modified maxPartitionBytes: {modified_partitions}")

Number of partitions with modified maxPartitionBytes: 457


In [23]:
spark.stop()